###Reference

For this project, the main tutorial I'll be using could be accessed through this link: https://medium.com/@o39joey/introduction-to-rag-with-python-langchain-62beeb5719ad. The information in this site will be used to create the basic RAG, but I'll add changes as I see fit.

In [ ]:
#installs important depedencies for this project
#chroma is the used vector database
#langchain is a framework for developing applications that use LLMs
#openai will be the LLM that's used
def download_dependencies():
  dependencies = [
      "langchain-chroma",
      "langchain-core==0.3.15",
      "langchain-community",
      "langchain-google-genai",
      "langchain-google-gemini",
      "langchain-text-splitters==0.3.2",
  ]


  for dependency in dependencies:
    !pip install "{dependency}"

In [ ]:
download_dependencies()

###Choice of LLM

For this project, I'll use gemini because it is cost effective and has good performance.

In [ ]:
#loading in gemini api key
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API')

###Embedder Function

This function loads the documents, creates them, and then stores them in the vector database.

In [ ]:
#function that embeds text
def embed_text(loader, splitter, vector_store):
  docs = loader.load()
  texts = splitter.create_documents([docs])
  return vector_store.add_documents(texts)


###Injecting Dependencies and Instantiating the Class

Because BrightBridge is an application that focuses on mental and relationship health, a good source to index would be "Intimate Relationships (9th Edition)."

Since I have it as a pdf, I'll be using Langchain's pdf loader.

The pdf content will also be split into 1000 character chunks to be individually embedded.

This initializes a google gemini embedder that converts the text into vector embeddings. Afterwards, a vector database (in this case, ChromaDB) is initialized. [0, 1]

In [ ]:
# imports
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter
from google.colab import userdata

# Parses the PDF
loader = PyPDFLoader("/content/Intimate Relationships (9th Edition with Contents).pdf")

# Initialize Text Splitter
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

# Get Embeddings Model
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

#connects the Chroma object to the cloud instance
vector_store = Chroma(
    collection_name="intimate_relationships_collection",
    embedding_function=embeddings,
    chroma_cloud_api_key=userdata.get("CHROMA_API_KEY"),
    tenant=userdata.get("CHROMA_TENANT"),
    database="Demo",
)

#embed text into the database
ids = embed_text(loader, text_splitter, vector_store)

###Testing the VectorDB



In [ ]:
# Query the Vector Store
results = vector_store.similarity_search(
    'What are some therapy options?',
    k=2
)

# Print Resulting Chunks
for res in results:
    print(f"* {res.page_content} [{res.metadata}]\n\n")

###Naive RAG Implementation

Here we will start implementing our naive RAG. [2, 3]

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

class Naive_RAG:
  #initializes naive rage with the gemini llm and the chromadb retriever
  def __init__(self, llm, retriever, prompt):

    # Create Prompt Instance from template
    custom_rag_prompt = PromptTemplate.from_template(prompt)

    self.rag_chain = (
    {"context": retriever | self.format_docs, "query": RunnablePassthrough()}
    | custom_rag_prompt
    | llm
    | StrOutputParser()
    )

  # Create Document Parsing Function to String
  def format_docs(self, docs):
      return "\n\n".join(doc.page_content for doc in docs)

  #invokes the prompt
  def invoke(self, prompt):
    return self.rag_chain.invoke(prompt)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

#gemini as llm
llm =  ChatGoogleGenerativeAI(model="gemini-2.5-flash",
                               api_key=GEMINI_API_KEY)

# Set Chroma Vector Store as the Retriever
retriever = vector_store.as_retriever()

# Create the Prompt Template
prompt_template = """Use the context provided to answer
the user's question below. If you do not know the answer
based on the context provided, tell the user that you do
not know the answer to their question based on the context
provided and that you are sorry.

context: {context}

question: {query}

answer: """

rag = Naive_RAG(llm, retriever, prompt_template)

In [ ]:
prompt = ""

rag.invoke(prompt)

###Langchain Documentation
0. https://docs.langchain.com/oss/python/integrations/embeddings/google_generative_ai

1.  Langchain Chromadb -- https://docs.langchain.com/oss/python/integrations/vectorstores/chroma

Langchain documentation w/ Gemini:

2.  https://docs.langchain.com/oss/python/langchain/rag#chroma
3.  https://reference.langchain.com/python/langchain-google-genai/chat_models/ChatGoogleGenerativeAI